# Run the processing pipeline

Builds the episode table (Stages 1–3) and runs download, transcription, ad removal and spontaneity check (Stages 4–7).

Run the cells from top to bottom. Transcription costs API credit, so do the smoke test before the full run.
The episode table and the processed table are saved in `data/`, so a later session can continue where the last one stopped.

## 1. Setup

In [4]:
import sys
import logging
from pathlib import Path

import pandas as pd

PROJECT_DIR = Path.cwd().parent
for folder in [PROJECT_DIR, PROJECT_DIR / "preprocessing"]:    # project root holds config.py
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

import config
from fetch_pipeline import FetchPipeline
from filter_index import PodcastDatabase
from extract_episodes import EpisodeFetcher
from download import EpisodeDownloader
from transcribe import EpisodeTranscriber
from remove_ads import AdRemover
from classify_spontaneity import SpontaneityClassifier
from processing_pipeline import ProcessingPipeline

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)

### Parameters

All settings of a run are in `config.py` in the project folder (years, groups and filters, `K`, `N_PODCASTS`, ad rule, models, paths).
Change them there, not here, and restart the kernel afterwards. This cell only shows the values of this run.

In [5]:
print(f"Years: {config.YEARS[0]}-{config.YEARS[-1]}, screening on {config.SCREENING_YEARS}")
print(f"Groups: {list(config.GROUPS)}")
print(f"N_PODCASTS = {config.N_PODCASTS}, K = {config.K}, K_TEST = {config.K_TEST}, BUFFER = {config.BUFFER}")
print(f"Excerpt: second {config.START_SEC} to {config.START_SEC + config.DURATION_SEC}, at least {config.MIN_WORDS_PER_EPISODE} words after ad removal")

Years: 2018-2026, screening on [2018, 2022]
Groups: ['first']
N_PODCASTS = 3, K = 2, K_TEST = 1, BUFFER = 2
Excerpt: second 300 to 900, at least 1000 words after ad removal


## 2. Build the episode table

Reads the RSS feeds of all matching podcasts, which takes a while. The table is saved and loaded from the file in later sessions.
Delete `data/episodes.pkl` to build it again, for example after changing `GROUPS`, `K`, `BUFFER` or `YEARS` in `config.py`.

In [6]:
if config.EPISODES_PATH.exists():
    episodes = pd.read_pickle(config.EPISODES_PATH)
else:
    tables = []
    for group, filters in config.GROUPS.items():
        pod_db = PodcastDatabase(str(config.DATABASE_PATH), config.DATABASE_TABLE)
        fetch_pipeline = FetchPipeline(pod_db, EpisodeFetcher(), group, filters=filters)
        tables.append(fetch_pipeline.build_podcast_df(k=config.K, buffer=config.BUFFER))
    episodes = pd.concat(tables, ignore_index=True)
    episodes.to_pickle(config.EPISODES_PATH)

print(episodes.groupby("group")["feed_url"].nunique().rename("candidate podcasts per group (pool, the pipeline takes the first N_PODCASTS that pass)"))
episodes.head()

group
first    4
Name: candidate podcasts per group (pool, the pipeline takes the first N_PODCASTS that pass), dtype: int64


,feed_url,podcast_title,episode_title,episode_year,episode_audio_url,episode_length,group,podcast_hash,episode_hash
0,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 77 - Fortnite Fiends,2018,https://anchor.fm/s/cc813770/podcast/play/6062...,3487.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,033a1f839d72626cbdebfbb8e6ed3d5b6e99322124e13e...
1,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 70 | E3 2018 Preshow,2018,https://anchor.fm/s/cc813770/podcast/play/6062...,5537.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,0f79c7a37cb224b49fa85c4c75d51c6886d1f27e961711...
2,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 67 | Is Anthem Doomed?,2018,https://anchor.fm/s/cc813770/podcast/play/6064...,6912.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,135ebed5ae6cb5fce75707ba8ac6ddb9fc38c1fb5387b5...
3,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 74 - The Downfall Of TellTale,2018,https://anchor.fm/s/cc813770/podcast/play/6062...,5106.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,654e432077f230e85ce0bcaad71e44acd20ee3dcc144a1...
4,https://feeds.feedburner.com/GamingIlluminaugh...,Gaming illuminaughty,Episode 68 | How Fortnite Took Over The World,2018,https://anchor.fm/s/cc813770/podcast/play/6063...,7184.0,first,51dd7e4e9f9b432995c335cb942bac9aa33fa20ff51d58...,6957e1dda5331a6e761f74b44edb5e0f1362decdc87777...


## 3. Create the pipeline

In [7]:
pipeline = ProcessingPipeline(
    ep_downloader=EpisodeDownloader(config.DOWNLOAD_DIR),
    ep_transcriber=EpisodeTranscriber(config.TRANSCRIPTION_DIR),
    ad_remover=AdRemover(config.AD_REMOVED_TRANSCRIPTION_DIR),
    classifier=SpontaneityClassifier(),
)

## 4. Smoke test

One podcast per group with one episode per year. Check the result and the cost before the full run.
Files written here are reused by the full run.

In [8]:
smoke = pipeline.process(episodes.copy(), k=1, n_podcasts=1, k_test=config.K_TEST)
smoke[smoke["episode_status"] != "pending"][["podcast_title", "podcast_status", "podcast_fail_reason", "episode_year", "episode_status", "episode_fail_reason",
                                             "n_words", "n_speakers", "score", "scripted"]]

2026-11-14 17:09:49,071 INFO Download OK: 17f1d33d9dd702bfd9c7fe092f44909ebbb017bbd5154fe5b5e7270719fe267d
2026-11-14 17:09:50,122 INFO Download OK: 033a1f839d72626cbdebfbb8e6ed3d5b6e99322124e13e40db03ebc6d44208c2
2026-11-14 17:09:50,126 INFO Starting transcription for 033a1f839d72626cbdebfbb8e6ed3d5b6e99322124e13e40db03ebc6d44208c2
2026-11-14 17:09:50,126 INFO Starting transcription for 17f1d33d9dd702bfd9c7fe092f44909ebbb017bbd5154fe5b5e7270719fe267d
2026-11-14 17:09:55,880 INFO HTTP Request: POST https://api.deepgram.com/v1/listen?diarize_model=v2&filler_words=true&model=nova-3&paragraphs=true&smart_format=true "HTTP/1.1 200 OK"
2026-11-14 17:09:56,003 INFO Writing to temp file /Users/lucasfreigang/PSL/Behavioral_Science_Applications/final_project/transcriptions/raw/17f1d33d9dd702bfd9c7fe092f44909ebbb017bbd5154fe5b5e7270719fe267d.part.json
2026-11-14 17:09:56,174 INFO HTTP Request: POST https://api.deepgram.com/v1/listen?diarize_model=v2&filler_words=true&model=nova-3&paragraphs=true

,podcast_title,episode_year,episode_status,episode_fail_reason,n_words,n_speakers,score,scripted
0,Gaming illuminaughty,2018,passed,<NA>,2333,2,4,False
12,Gaming illuminaughty,2019,passed,<NA>,2284,4,4,False
28,Gaming illuminaughty,2020,passed,<NA>,2304,2,4,False
44,Gaming illuminaughty,2021,passed,<NA>,2255,4,4,False
53,Gaming illuminaughty,2022,passed,<NA>,2234,2,4,False
71,Gaming illuminaughty,2023,passed,<NA>,2259,3,2,True
83,Gaming illuminaughty,2024,passed,<NA>,2268,3,4,False
95,Gaming illuminaughty,2025,passed,<NA>,2064,3,4,False
104,Gaming illuminaughty,2026,passed,<NA>,2347,4,4,False


## 5. Full run

Continues from `data/processed.pkl` if it exists, so nothing is processed twice.
If the run is interrupted, run this cell again: finished downloads, transcripts and ad files are reused from disk.

In [9]:
start = pd.read_pickle(config.PROCESSED_PATH) if config.PROCESSED_PATH.exists() else episodes.copy()

processed = pipeline.process(start, k=config.K, n_podcasts=config.N_PODCASTS, k_test=config.K_TEST)
processed.to_pickle(config.PROCESSED_PATH)

2026-11-14 18:08:32,790 INFO Download OK: 003bb81f823324f186981a32885927d7e4310167739c3a3c92cd518dea6ca919
2026-11-14 18:08:33,194 INFO Download OK: 017ea2140940f56a399b2cd8753f6667d7ccc7cacc7a254da81177d4194e1996
2026-11-14 18:08:33,492 INFO Download OK: 080acc77f27b81afd5975b9bc32375281a8da0a79c09aa8992cc748580b26967
2026-11-14 18:08:33,939 INFO Download OK: 016a6e40c1e9363ad881a508a94f5c19695b2283d651f1ede47be03aefb6f929
2026-11-14 18:08:33,943 INFO Starting transcription for 016a6e40c1e9363ad881a508a94f5c19695b2283d651f1ede47be03aefb6f929
2026-11-14 18:08:33,943 INFO Starting transcription for 080acc77f27b81afd5975b9bc32375281a8da0a79c09aa8992cc748580b26967
2026-11-14 18:08:33,943 INFO Starting transcription for 017ea2140940f56a399b2cd8753f6667d7ccc7cacc7a254da81177d4194e1996
2026-11-14 18:08:33,943 INFO Starting transcription for 003bb81f823324f186981a32885927d7e4310167739c3a3c92cd518dea6ca919
2026-11-14 18:08:41,142 INFO HTTP Request: POST https://api.deepgram.com/v1/listen?diari

## 6. Results

Podcasts per group and status. `pending` podcasts were not needed.

In [10]:
podcasts = processed.drop_duplicates("feed_url")
podcasts.groupby(["group", "podcast_status"]).size().unstack(fill_value=0)

podcast_status,passed,pending
group,,
first,3,1


Processed episodes per podcast and year. Every cell of a passed podcast should hold `K`.

In [12]:
done = processed[(processed["podcast_status"] == "passed") & (processed["episode_status"] == "processed")]
done.groupby(["group", "podcast_title", "episode_year"]).size().unstack(fill_value=0)

,episode_year
group,podcast_title


Why podcasts and episodes failed.

In [13]:
print(podcasts[podcasts["podcast_status"] == "failed"]["podcast_fail_reason"].value_counts(), "\n")

failed = processed[processed["episode_status"] == "failed"]
print(failed["fail_stage"].value_counts())
failed[["podcast_title", "episode_year", "fail_stage", "episode_fail_reason"]]

Series([], Name: count, dtype: int64) 

fail_stage
spontaneity    30
Name: count, dtype: int64


,podcast_title,episode_year,fail_stage,episode_fail_reason
83,Gaming illuminaughty,2024,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
96,Gaming illuminaughty,2025,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
104,Gaming illuminaughty,2026,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
105,Gaming illuminaughty,2026,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
175,Rand(Nerds);,2019,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
176,Rand(Nerds);,2019,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
200,Rand(Nerds);,2020,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
227,Rand(Nerds);,2021,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
278,Rand(Nerds);,2023,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."
279,Rand(Nerds);,2023,spontaneity,"RateLimitError(""Error code: 429 - {'error': {'..."


Spontaneity score and transcription quality per group and year (processed episodes of passed podcasts).

In [14]:
done.astype({"score": float, "mean_confidence": float, "n_words": float}).groupby(["group", "episode_year"])[["score", "mean_confidence", "n_words"]].mean().round(2)

,,score,mean_confidence,n_words
group,episode_year,,,
